# 00 環境と基準 — 「正解」を知っている迷路で、強化学習の準備をする

## ゴール
- 強化学習の基本の言葉（**エージェント・環境・状態・行動・報酬**）を、小さな迷路で理解する。
- **ランダムに動くとどうなるか**（下限）と、**最適な動き方**（動的計画法で厳密に求めた正解）を知る。
- 以降のNotebookで、学習した方策を、**この基準と比べて**評価できるようにする。

## 強化学習とは
分類や予測（教師あり学習）では、「この入力にはこの答え」という**正解**が与えられます。
強化学習では、**正解の行動は教えてもらえません**。エージェントが**行動**して、環境から**報酬**（良し悪しの点数）だけが返ってきます。
その報酬の合計（**リターン**）を、できるだけ大きくする行動の仕方（**方策**）を、**試行錯誤で見つけます**。

```
        行動 a
 ┌─────────────▶┐
 エージェント      環境（迷路）
 └◀─────────────┘
   状態 s・報酬 r
```

## キーワード
- **状態 s**: 今の状況（ここでは、迷路のどのマスにいるか）。
- **行動 a**: できること（上・右・下・左）。
- **報酬 r**: 行動の結果として返る点数。ここでは、**1歩ごとに −1**（早くゴールするほど合計が大きい）。崖に落ちると −100。
- **リターン**: 1回の挑戦（エピソード）での、報酬の合計。**これを大きくするのが目的**。
- **方策**: 状態から行動を決める規則。
- **エピソード**: スタートからゴール（または打ち切り）までの1回の挑戦。
- **状態価値 V**: ある状態から、ある方策に従ったときの、リターンの見込み。
- **動的計画法（価値反復）**: 環境の仕組み（遷移の確率）を**知っている**ときに、最適な価値・方策を**厳密に**求める方法。学習するエージェントは仕組みを知らないので、これは「**正解を知る側**」の道具です。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from rl_envs import (LAYOUTS, ACTION_NAMES, make_env, random_policy, value_iteration, policy_evaluation,
                     evaluate_policy, shortest_path_length)
from rl_envs.viz import draw_grid

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, name in zip(axes, ["room", "maze", "cliff"]):
    env = make_env(name)
    draw_grid(env, ax, title=f"{name}（状態数 {env.n_states}）")
fig.tight_layout()

**何をしているか**: 3つの迷路を描きます。**S**がスタート、**G**がゴール、黒が壁、赤が**崖**（落ちるとスタートに戻され、大きな罰）です。

**出力の見方**:
- **room**: 壁のない広い部屋。S→Gは最短8歩。
- **maze**: 壁のある迷路。ほぼ一本道で、最短12歩。
- **cliff**: 下の段が崖。**崖沿いの最短ルート**（13歩）と、**上を回る安全なルート**がある。後で、学習方法の違いが出る場面です。

In [ ]:
env = make_env("room")
s = env.reset(seed=0)
print("状態の数:", env.n_states, " 行動:", ACTION_NAMES)
print("スタートの状態:", s, "= マス", env.state_to_cell[s])
for a in [2, 2, 1]:                            # 下、下、右 と動いてみる
    s2, r, done, truncated = env.step(a)
    print(f"行動 {ACTION_NAMES[a]} → 状態 {s2} = マス {env.state_to_cell[s2]}, 報酬 {r}, ゴール到達 {done}")

**何をしているか**: 環境を1歩ずつ動かします。`step(a)` が「行動 → 次の状態・報酬・終了か」を返します。**学習するエージェントが使うのは、この入口だけ**です（迷路の仕組みは教えられません）。

**出力の見方**: 下・下・右と動くと、マスが (0,0) → (1,0) → (2,0) → (2,1) と動き、1歩ごとに報酬 −1 が返ります。

In [ ]:
res = []
for name in ["room", "maze", "cliff"]:
    e = make_env(name)
    rnd = random_policy(e)     # どのマスでも、4つの行動を同じ確率で選ぶ（迷路ごとに、状態の数が違うので作り直す）
    r = evaluate_policy(e, rnd, n_episodes=1000, seed=1_000_000)
    res.append(dict(迷路=name, 平均リターン=round(r["mean_return"], 1), ゴール到達率=r["success_rate"], 平均歩数=round(r["mean_steps"], 1)))
pd.DataFrame(res).set_index("迷路")

**何をしているか**: 何も学習していない**ランダム方策**で、各迷路を1000回挑戦します（1回の上限は200歩で、打ち切ります）。

**出力の見方**:
- **room**: 平均リターン約 −91。ゴール到達率87%。広い部屋なので、うろうろしていればたいてい着くが、ひどく遠回り（平均91歩。最短は8歩）。
- **maze**: 到達率22%。迷路が狭く、**ほとんどの挑戦で200歩のうちにゴールできない**。
- **cliff**: 到達率2%、平均リターン約 −2096。**何度も崖に落ちる**（1回−100）。
- これが**学習の出発点（下限）**です。強化学習は、ここからどれだけ良くなるかで評価します。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
rows = []
for ax, name in zip(axes, ["room", "maze", "cliff"]):
    e = make_env(name)
    V, pi = value_iteration(e)                 # 動的計画法で、最適な価値と方策を厳密に求める
    draw_grid(e, ax, V=V, policy=pi, title=f"{name}: 最適方策（矢印）と状態価値（色）", show_values=False)
    r = evaluate_policy(e, pi, n_episodes=100)
    rows.append(dict(迷路=name, 最適な価値_スタート=round(V[e.start], 2), 最短歩数=shortest_path_length(e),
                     実際に動かした平均リターン=r["mean_return"], ゴール到達率=r["success_rate"]))
fig.tight_layout()
pd.DataFrame(rows).set_index("迷路")

**何をしているか**: **価値反復**で、各迷路の**最適方策**（各マスでの最良の行動、矢印）と**状態価値**（色。明るいほどゴールに近く、リターンが大きい）を求めます。そして、その方策を実際に動かします。

**出力の見方**:
- 矢印をたどると、S→Gの**最短ルート**になっています。cliff では、**崖沿いを進む**ルートです。
- スタートの価値は、最短歩数のマイナス（room −8、maze −12、cliff −13）と**一致**します。床が滑らない（確率的でない）ときは、最適なリターン ＝ −最短歩数です。
- 実際に動かすと、毎回同じ（平均リターンが価値と同じで、到達率1.0）です。
- これが**正解**です。以降、学習した方策がこの値にどれだけ近づくかを見ます。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.6))
rows = []
for i, name in enumerate(["room", "cliff"]):
    for j, slip in enumerate([0.0, 0.2]):
        e = make_env(name, slip=slip)
        V, pi = value_iteration(e)
        draw_grid(e, axes[i][j], V=V, policy=pi, title=f"{name}（すべる確率 {slip}）: 最適な価値 {V[e.start]:.2f}")
        rows.append(dict(迷路=name, すべる確率=slip, 最適な価値=round(V[e.start], 2),
                         実際に動かした平均リターン=round(evaluate_policy(e, pi, 500)["mean_return"], 2)))
fig.tight_layout()
pd.DataFrame(rows).set_index(["迷路", "すべる確率"])

**何をしているか**: 床が**すべる**（選んだ行動の代わりに、確率0.2で左右に滑る）場合の最適方策を求めます。

**出力の見方**:
- **room**: すべる確率0.2で、最適な価値が −8 → −9.81 に下がります（遠回りになる）。矢印は大きくは変わりませんが、右へ進む代わりに**下へ進むマスが増えます**（壁のある右端に沿うほうが、滑っても外に出ないため）。
- **cliff**: **価値が −13 → −29.77 と大きく下がり、方策が変わります**。滑って崖に落ちると −100 なので、**崖から離れた上の2段を進む**ようになります（崖のすぐ上の段は、↑で上へ戻ります）。
  さらに、**スタートでは「上」ではなく「左」**を選びます。左は壁なので、押し付けてとどまり、確率0.1で上へ滑って進めるだけです。**右の崖へ滑る危険が、全くない**ためで、何歩か待つ代わりに −100 を避けています。学習するエージェントが、壁を使うこの賢い動きを、試行錯誤で見つけられるかどうかも、見どころです。
- 同じ迷路でも、**環境の性質（すべる・すべらない）で最適な行動が変わります**。学習するエージェントは、これを試行錯誤で見つけなければなりません。

In [ ]:
e = make_env("room")
V_exact = policy_evaluation(e, random_policy(e))[e.start]        # 打ち切りなしの厳密な価値
V_sim = evaluate_policy(e, random_policy(e), 1000)["mean_return"] # 実際に200歩で打ち切って動かした平均
print(f"ランダム方策の価値（厳密・打ち切りなし）: {V_exact:.1f}")
print(f"ランダム方策の平均リターン（シミュレーション・200歩で打ち切り）: {V_sim:.1f}")

**何をしているか**: ランダム方策の、**厳密な価値**と、**実際に動かした平均**を比べます。

**出力の見方**: 厳密な価値（約 −107）は、シミュレーションの平均（約 −91）と**違います**。
シミュレーションでは**200歩で打ち切る**ため、うろうろして長引いた挑戦のリターンが、−200 でおさまるからです。
**評価の条件（打ち切りの有無）が、数字を変える**ことを、頭に入れておきましょう。以降は、すべて**打ち切りあり**の条件（`max_steps=200`）で、シミュレーションして比べます。

## 試してみよう（演習）
1. **迷路を変える**: `make_env("maze")` の `"maze"` を `"room"` に変えて、ランダム方策の到達率と平均歩数を比べてください。広い部屋は偶然ゴールしやすく、迷路は難しいことが分かります。
2. **すべる確率を変える**: `make_env("cliff", slip=0.2)` の `0.2` を `0.05` と `0.4` に変えて、最適方策の矢印（崖からどれだけ離れるか）がどう変わるか見てください。分かるのは、滑りやすいほど、安全なルートを選ぶ（遠回りでも）ということです。
3. **報酬を変える**: `make_env("maze", step_reward=-0.1, goal_reward=10)` にして、最適な価値を求めてください。分かるのは、**報酬の設計**（1歩ごとの罰とゴールの賞金）を変えても、最適な**ルート**は変わらないが、**価値の大きさ**は変わるということです（報酬ハッキングは、Phase3で扱います）。
4. **自分の迷路**: `LAYOUTS["maze"]` を参考に、文字列のリストで迷路を作って `GridWorld(layout)` に渡してください（`S` と `G` が必要）。

## まとめ
- 強化学習は、**正解の行動を教えられず**、**報酬から**良い方策を見つける。
- **ランダム方策**は、迷路が難しいほど（maze・cliff）全くゴールできない。これが学習の**下限**。
- 環境の仕組みを知っていれば、**動的計画法**で**最適方策を厳密に**求められる。これが**正解**。学習するエージェントは、仕組みを知らずに、これに近づこうとする。
- 床が滑る・崖があるなど、**環境の性質**で、最適な行動が変わる。
- 評価は、**打ち切りの有無**などの条件で数字が変わる。**条件をそろえて**比べる。

次の Phase1 では、状態のない最も簡単な問題（**多腕バンディット**）で、強化学習の核心である**探索と活用**を調べます。